<a href="https://colab.research.google.com/github/yuri-maradini/TempSal/blob/main/src/train_ueyes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning TempSAL su UEyes — Colab

Notebook pronto per lanciare il training vero (Step 4) su GPU, invece che sulla CPU locale.

**Prima di eseguire questo notebook**, su Google Drive crea una cartella (default atteso: `MyDrive/TempSAL_UEyes/`) contenente:
- `multilevel_tempsal.pt` — il checkpoint pre-addestrato originale
- `data_ueyes.zip` — l'archivio di `data_ueyes/` generato in locale (Step 1-2)

Poi: **Runtime → Cambia tipo di runtime → GPU**, prima di eseguire le celle.

In [1]:
# Controllo che sia stata assegnata una GPU
!nvidia-smi

Mon Sep 14 16:00:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Cambia questo path se hai usato un nome/percorso diverso su Drive
DRIVE_DIR = '/content/drive/MyDrive/TempSAL_UEyes'

assert os.path.isdir(DRIVE_DIR), (
    f"Cartella non trovata: {DRIVE_DIR}\n"
    "Creala su Drive e caricaci multilevel_tempsal.pt + data_ueyes.zip prima di continuare."
)
print('Contenuto trovato su Drive:', os.listdir(DRIVE_DIR))

Contenuto trovato su Drive: ['multilevel_tempsal.pt', 'data_ueyes.zip', 'multilevel_tempsal_ueyes.pt', 'multilevel_tempsal_ueyes_v2.pt', 'multilevel_tempsal_ueyes_v3.pt', 'multilevel_tempsal_ueyes_v4.pt', 'multilevel_tempsal_ueyes_v5.pt', 'multilevel_tempsal_ueyes_v6.pt']


In [4]:
# Codice: sempre aggiornato da GitHub, non serve preparazione
!git clone https://github.com/yuri-maradini/TempSal.git /content/TempSal

Cloning into '/content/TempSal'...
remote: Enumerating objects: 309, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 309 (delta 136), reused 240 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (309/309), 14.84 MiB | 17.61 MiB/s, done.
Resolving deltas: 100% (136/136), done.


In [5]:
!cd /content/TempSal && git pull


Already up to date.


In [6]:
import shutil

os.makedirs('/content/TempSal/src/checkpoints', exist_ok=True)
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal.pt',
)
# multilevel_tempsal_ueyes_v4.pt e' il checkpoint della quarta run (backbone
# sbloccato, statistiche BatchNorm congelate, il migliore sul ramo temporale
# finora): questa run riparte da li'.
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v4.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v4.pt',
)
print('Checkpoint copiati (originale + v4, il warm-start di questa run).')

Checkpoint copiati (originale + v4, il warm-start di questa run).


In [7]:
import time
import zipfile

# Estratto sul disco locale di Colab (veloce), non lasciato sul mount di Drive
# (l'I/O su Drive montato e' molto piu' lento per tanti file piccoli, e qui
# ce ne sono migliaia tra immagini, mappe e volumi temporali).
t0 = time.time()
with zipfile.ZipFile(f'{DRIVE_DIR}/data_ueyes.zip') as zf:
    zf.extractall('/content/TempSal/')
print(f'Dati estratti in {time.time() - t0:.0f}s')

Dati estratti in 32s


In [8]:
# Controllo veloce di integrita': i conteggi devono combaciare con quelli
# verificati in locale (1872 train / 108 val per ciascuna sottocartella)
for sub in ['images', 'maps', 'fixation_maps', 'saliency_volumes_5', 'fixation_volumes_5']:
    for split in ['train', 'val']:
        d = f'/content/TempSal/data_ueyes/{sub}/{split}'
        n = len(os.listdir(d)) if os.path.isdir(d) else 'MANCANTE'
        print(f'{sub:22s} {split:5s} -> {n}')

images                 train -> 1872
images                 val   -> 108
maps                   train -> 1872
maps                   val   -> 108
fixation_maps          train -> 1872
fixation_maps          val   -> 108
saliency_volumes_5     train -> 9360
saliency_volumes_5     val   -> 540
fixation_volumes_5     train -> 9360
fixation_volumes_5     val   -> 540


In [9]:
# Colab ha gia' PyTorch con supporto CUDA preinstallato: installiamo solo le
# altre dipendenze del progetto, senza toccare torch/torchvision/torchaudio
# (forzare i pin usati in locale, pensati per una build CPU, rischierebbe di
# rimpiazzare la build CUDA gia' pronta di Colab con una incompatibile).
!pip install -q wandb pycocotools ftfy einops clip-anytorch kornia regex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 85.1 MB/s eta 0:00:00


In [10]:
# Verifica che l'installazione sopra non abbia rovinato il supporto CUDA di torch
import torch
print('torch', torch.__version__, '| CUDA disponibile:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU non disponibile: controlla Runtime > Cambia tipo di runtime > GPU'

torch 2.11.0+cu128 | CUDA disponibile: True


## wandb (consigliato per questa run)

Il primo run (10 epoche) è stato fatto con `WANDB_MODE=disabled`: nessuna curva salvata, i numeri per epoca sono stati recuperati a mano dall'output della cella di training. Per questa run vale la pena accendere il logging vero, così le curve di CC/KLDIV/NSS/SIM (aggregate e per-slice) restano disponibili per il confronto e per la tesi senza dover rileggere l'output della cella.

Esegui la cella sotto (chiede l'API key, la trovi su wandb.ai/authorize) prima di lanciare il training. Se preferisci comunque saltarlo, aggiungi di nuovo `WANDB_MODE=disabled` (o `=offline` per salvare i log in locale senza account) davanti al comando `python train.py` nella cella di training.</cell id="HNcrR_LSgcXM">


In [11]:
import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yurimaradini (yurimaradini-universit-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Training (run 7 — sblocco anche di `pnas_sal`)

**Risultati delle run 3-6**: sbloccare il backbone di `pnas_vol` (con le statistiche BatchNorm congelate) ha aiutato in modo netto il ramo temporale, ma la mappa aggregata è rimasta ferma al livello di `v2` in ogni configurazione provata da allora — né più epoche, né un learning rate differenziato sul mixing decoder (a 3× o 10× `--lr`) l'hanno spostata. In tutte queste run **`pnas_sal`, il ramo che produce direttamente la mappa aggregata, è rimasto sempre completamente congelato** (pesi e statistiche BatchNorm), fin dalla prima run: è l'unica leva rilevante mai testata finora.

**Questa run sblocca anche `pnas_sal`**, seguendo esattamente lo stesso schema già validato per `pnas_vol` nelle run 3/4 — non un'idea nuova, la stessa idea applicata all'altro ramo:
- Nuovo argomento in `train.py`, `--train_sal_enc`: sblocca `pnas_sal` per intero (backbone + la sua testa `deconv_layer0..5`) come un unico blocco — a differenza di `pnas_vol`, `pnas_sal` non è mai stato pensato con un controllo separato backbone/testa, quindi non esiste un caso "solo testa" da preservare qui. Default `0`, comportamento identico a tutte le run precedenti se non passato.
- **Il fix per le statistiche BatchNorm è applicato fin da subito**, non scoperto a posteriori come nello Step 4.5: `freeze_batchnorm_stats()` (già scritta per `pnas_vol`) viene riusata sul backbone di `pnas_sal` quando `--train_sal_enc 1`, cosicché `running_mean`/`running_var` restino ancorati ai valori di `v4` mentre i pesi si aggiornano — nessuno shock iniziale da BatchNorm atteso stavolta.
- **Un solo cambiamento rispetto a `v4`**: stesso comando di `v4` (`--train_enc 1 --train_model 1 --lr 1e-6`, niente `--mixing_lr` esplicito — ricade su `--lr` esattamente come in `v4`), con l'aggiunta di `--train_sal_enc 1`. Isola l'effetto della nuova variabile senza confonderlo con altri cambiamenti.
- Warm-start da `multilevel_tempsal_ueyes_v4.pt` (il checkpoint finale finora).
- `--batch_size 8 --grad_accum_steps 4` (batch effettivo 32, invariato): con **due** backbone PNAS interi ora sotto backprop invece di uno solo, la memoria per singolo forward/backward raddoppia di nuovo rispetto a v3-v6 — stesso tipo di problema OOM incontrato nello Step 4.5, stavolta anticipato invece di scoperto lanciando la run.
- `--no_epochs 10`, stesso budget usato per testare lo sblocco di `pnas_vol` in v3/v4.

**Verificato in locale su CPU** prima di lanciare su Colab (mini-dataset, warm-start da `v4`, `--train_enc 1 --train_sal_enc 1`, 2 epoche): confronto peso-per-peso conferma che `pnas_sal` impara davvero (789/791 tensori cambiati) mentre le sue statistiche BatchNorm restano fisse (0/603 invariate) — lo stesso pattern già verificato per `pnas_vol` nello Step 4.6, stavolta sull'altro ramo. `pnas_vol` continua a comportarsi come nelle run precedenti (775/777 pesi cambiati, 0/603 statistiche invariate), il mixing decoder resta 14/14 allenabile: nessuna regressione introdotta.

**Da controllare dopo la run**: se la mappa aggregata (CC/KLDIV di validazione) supera finalmente il livello di `v2`/`v4` → sbloccare `pnas_sal` è la leva che mancava. Se resta comunque piatta o peggiora → anche l'ultima leva "dentro" l'architettura di TempSAL è stata esclusa, e resta solo la dimensione del dataset (o un cambio di architettura) come possibile spiegazione residua per il gap con `v2`.

In [12]:
%cd /content/TempSal/src
# PYTORCH_CUDA_ALLOC_CONF: riduce la frammentazione dell'allocatore, utile
# ora che la memoria e' molto piu' vicina al limite della T4 (vedi cella
# markdown sulla run 4 sull'OOM del primo tentativo).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py \
  --enc_model pnas_boosted_multi \
  --dataset_dir ../data_ueyes/ \
  --model_path ./checkpoints/multilevel_tempsal_ueyes_v4.pt \
  --model_vol_path ./checkpoints/multilevel_tempsal_ueyes_v4.pt \
  --train_model 1 \
  --train_enc 1 \
  --train_sal_enc 1 \
  --lr 1e-6 \
  --batch_size 8 \
  --grad_accum_steps 4 \
  --no_epochs 10 \
  --model_val_path ./checkpoints/multilevel_tempsal_ueyes_v7.pt

/content/TempSal/src
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: yurimaradini (yurimaradini-universit-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/TempSal/src/wandb/run-20260914_160352-p9ab1c7r
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run wise-deluge-9
wandb: ⭐️ View project at https://wandb.ai/yurimaradini-universit-di-padova/tempsal-ueyes
wandb: 🚀 View run at https://wandb.ai/yurimaradini-universit-di-padova/tempsal-ueyes/runs/p9ab1c7r
PNAS Boosted Model PNASBoostedModelMultiLevel
96 96 54
96 270 108
270 540 216
540 1080 216
1080 1080 216
1080 1080 216
1080 1

In [13]:
# Copia il checkpoint fine-tuned su Drive, cosi' sopravvive alla chiusura
# della sessione Colab. Puoi rieseguire questa cella anche a training ancora
# in corso, per avere un backup intermedio.
import shutil

src_ckpt = '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v7.pt'
dst_ckpt = f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v7.pt'
shutil.copy(src_ckpt, dst_ckpt)
print('Copiato su Drive:', dst_ckpt)

Copiato su Drive: /content/drive/MyDrive/TempSAL_UEyes/multilevel_tempsal_ueyes_v7.pt
